# Face2Comic — pix2pix Hyperparameter Tuning & Training

This notebook is a **self-contained** pipeline for training a pix2pix conditional GAN that converts real face photos into comic-style images.

**What's inside:**
1. Data loading (pre-processed `.npy` arrays, already normalized to [-1, 1])
2. U-Net Generator + PatchGAN Discriminator (built from scratch)
3. Hyperparameter grid search over learning rate, batch size, and L1 weight
4. Visualization of grid search results (charts, heatmaps, sample grids)
5. Full training run with the best configuration
6. Training curves, sample progression, and final evaluation

In [ ]:
import sys
sys.path.insert(0, "..")

import csv
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from torchinfo import summary

from src.config import SEED, NUM_WORKERS, set_seed, get_device
from src.data import load_data, make_datasets, make_loaders
from src.models import Generator, Discriminator
from src.training import make_models, train_one_epoch, validate
from src.visualization import denorm, save_grid_samples

set_seed(SEED)
print(f"Random seed: {SEED}")
print(f"Number of workers: {NUM_WORKERS}")

## Device Selection

Use CUDA if available, otherwise fall back to MPS (Apple Silicon) or CPU.

In [ ]:
DEVICE = get_device()
print(f"Device: {DEVICE}")

## 1. Load Data

The preprocessing notebook already resized all images to 256x256, converted them to CHW format, and normalized pixel values to [-1, 1]. We load them as memory-mapped arrays so only the batches we need are read into RAM.

In [ ]:
data = load_data("../data/npy")

print(f"Train — real: {data['train_real'].shape}, comic: {data['train_comic'].shape}")
print(f"Val   — real: {data['val_real'].shape},   comic: {data['val_comic'].shape}")
print(f"Test  — real: {data['test_real'].shape},  comic: {data['test_comic'].shape}")
print(f"Pixel range: [{data['train_real'][0].min():.1f}, {data['train_real'][0].max():.1f}]")

## 2. Dataset & DataLoader

A thin wrapper that converts memory-mapped NumPy rows into PyTorch tensors on the fly. For the grid search we use a **2 000-sample training subset** and a **500-sample validation subset** to keep each trial fast (~2 min). The full training and validation splits are used for final model training, and the held-out test split is reserved for final evaluation only.

In [ ]:
datasets = make_datasets(data)

train_dataset = datasets["train"]
full_val_dataset = datasets["val"]
full_test_dataset = datasets["test"]
train_tuning_dataset = datasets["train_tuning"]
val_tuning_dataset = datasets["val_tuning"]

print(f"Train samples (full): {len(train_dataset)}")
print(f"Train samples (tuning subset): {len(train_tuning_dataset)}")
print(f"Val samples (full): {len(full_val_dataset)}")
print(f"Val samples (tuning subset): {len(val_tuning_dataset)}")
print(f"Test samples (full): {len(full_test_dataset)}")

## 3. Generator — U-Net

The generator is an encoder-decoder with skip connections. The encoder compresses the 256x256 input down to a 4x4 bottleneck, and the decoder upsamples it back. Skip connections carry high-frequency spatial detail (edges, facial features) from encoder to decoder so they survive the bottleneck.

- **Encoder blocks**: Conv2d(stride=2) → InstanceNorm → LeakyReLU
- **Decoder blocks**: ConvTranspose2d(stride=2) → InstanceNorm → ReLU (+ Dropout in first two)
- **Output**: Tanh activation → pixel values in [-1, 1]

In [ ]:
generator = Generator()
summary(generator, input_size=(1, 3, 256, 256))

## 4. Discriminator — PatchGAN

In [ ]:
discriminator = Discriminator()
summary(discriminator, input_size=[(1, 3, 256, 256), (1, 3, 256, 256)])

### Weight Initialization & Parameter Count

Standard pix2pix initialization: draw conv weights from N(0, 0.02) and set BatchNorm weights to N(1, 0.02) with zero bias.

In [ ]:
_g = Generator()
_d = Discriminator()
print(f"Generator params:     {sum(p.numel() for p in _g.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in _d.parameters()):,}")
del _g, _d

## 5. Loss Functions

Two losses work together to make generated comics as close as possible to the ground truth:

| Loss | Purpose |
|------|--------|
| **BCE (adversarial)** | Fools the discriminator — makes outputs look "real" |
| **L1 (pixel)** | Penalizes pixel-level deviation from the target comic |

The generator loss is: `G_loss = BCE + λ_L1 × L1`

## 6. Hyperparameter Grid Search

We search over **3 learning rates × 3 batch sizes × 3 L1 weights = 27 configurations**. Each one trains for 10 short epochs on a **2 000-sample training subset** and is validated on the 500-sample tuning subset — this keeps each trial fast (~2 min).

The CSV log is written after every config so nothing is lost if the kernel dies. The best config (lowest validation L1) gets its checkpoint saved.

In [ ]:
GRID_LRS = [1e-4, 2e-4, 5e-4]
GRID_BATCH_SIZES = [16, 32, 64]
GRID_LAMBDA_L1 = [50, 100, 150]
GRID_EPOCHS = 10
GRID_BETAS = (0.5, 0.999)
GRID_PREVIEW_INDICES = [9, 10, 12]

OUTPUT_DIR = Path("output/grid_search_samples+3_hyperparameter_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path("output/grid_search_results+3_hyperparameter_tuning")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

total_configs = len(GRID_LRS) * len(GRID_BATCH_SIZES) * len(GRID_LAMBDA_L1)
print(f"Grid search: {total_configs} configurations, {GRID_EPOCHS} epochs each")

In [ ]:
GRID_CSV = RESULTS_DIR / "grid_search_results.csv"

with open(GRID_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["config_idx", "learning_rate", "batch_size", "lambda_l1", "val_l1"])
    writer.writeheader()

grid_results = []
best_val_l1 = float("inf")
best_grid_state = None
config_idx = 0

for lr in GRID_LRS:
    for bs in GRID_BATCH_SIZES:
        for lam in GRID_LAMBDA_L1:
            config_idx += 1
            print(f"\n{'='*60}")
            print(f"Config {config_idx}/{total_configs}  LR={lr:.0e}  BS={bs}  λ_L1={lam}")
            print(f"{'='*60}")

            gen, disc, opt_gen, opt_disc = make_models(lr, GRID_BETAS, DEVICE)
            train_loader, val_loader = make_loaders(bs, train_tuning_dataset, val_tuning_dataset, DEVICE)

            for epoch in range(1, GRID_EPOCHS + 1):
                g_loss, d_loss = train_one_epoch(gen, disc, train_loader, opt_gen, opt_disc, lam, DEVICE)
                if epoch % 5 == 0 or epoch == GRID_EPOCHS:
                    print(f"  Epoch {epoch}/{GRID_EPOCHS}  G={g_loss:.4f}  D={d_loss:.4f}")

            val_l1 = validate(gen, val_loader, DEVICE)
            print(f"  Val L1: {val_l1:.6f}")

            save_grid_samples(gen, full_val_dataset, config_idx, OUTPUT_DIR, DEVICE, GRID_PREVIEW_INDICES)

            # show a quick 3-sample preview inline
            preview_indices = [idx for idx in GRID_PREVIEW_INDICES if 0 <= idx < len(full_val_dataset)]
            if not preview_indices:
                preview_indices = list(range(min(3, len(full_val_dataset))))

            gen.eval()
            fig, axes = plt.subplots(len(preview_indices), 3, figsize=(9, 3 * len(preview_indices)), squeeze=False)
            axes[0, 0].set_title("Real")
            axes[0, 1].set_title("Generated")
            axes[0, 2].set_title("Target")
            with torch.no_grad():
                for row, idx in enumerate(preview_indices):
                    real_s, target_s = full_val_dataset[idx]
                    fake_s = gen(real_s.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()
                    axes[row, 0].imshow(denorm(real_s).permute(1, 2, 0).numpy())
                    axes[row, 1].imshow(denorm(fake_s).permute(1, 2, 0).numpy())
                    axes[row, 2].imshow(denorm(target_s).permute(1, 2, 0).numpy())
                    for j in range(3):
                        axes[row, j].axis("off")
            fig.suptitle(f"Config {config_idx}  LR={lr:.0e}  BS={bs}  λ={lam}  Val L1={val_l1:.4f}", fontweight="bold")
            plt.tight_layout()
            plt.show()

            row = {"config_idx": config_idx, "learning_rate": lr, "batch_size": bs, "lambda_l1": lam, "val_l1": round(val_l1, 6)}
            grid_results.append(row)

            with open(GRID_CSV, "a", newline="") as f:
                csv.DictWriter(f, fieldnames=row.keys()).writerow(row)

            if val_l1 < best_val_l1:
                best_val_l1 = val_l1
                best_grid_state = {
                    "config_idx": config_idx,
                    "lr": lr, "bs": bs, "lambda_l1": lam,
                    "val_l1": val_l1,
                    "gen_state_dict": gen.state_dict(),
                    "disc_state_dict": disc.state_dict(),
                }

            del gen, disc, opt_gen, opt_disc
            torch.cuda.empty_cache()

# save best grid checkpoint
torch.save(best_grid_state, CHECKPOINT_DIR / "best_grid_config.pt")

print(f"\n{'='*60}")
print("Grid Search Complete!")
print(f"{'='*60}")
print(f"\nBest Config #{best_grid_state['config_idx']}  (Val L1 = {best_val_l1:.6f})")
print(f"  LR:       {best_grid_state['lr']:.0e}")
print(f"  Batch:    {best_grid_state['bs']}")
print(f"  λ_L1:     {best_grid_state['lambda_l1']}")
print(f"\nResults saved to {GRID_CSV}")

## 7. Grid Search — Visual Analysis

In [ ]:
cfg = [r["config_idx"] for r in grid_results]
vals = [r["val_l1"] for r in grid_results]
best_i = vals.index(min(vals))

plt.figure(figsize=(10, 4))
plt.plot(cfg, vals, marker="o", linewidth=1.5)
plt.scatter([cfg[best_i]], [vals[best_i]], color="red", s=70, zorder=3, label="Best")
plt.title("Validation L1 by Configuration")
plt.xlabel("Configuration #")
plt.ylabel("Validation L1")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "val_l1_by_config.png", dpi=100)
plt.show()

In [ ]:
# Bar chart: Val L1 per config, colored by LR
sorted_results = sorted(grid_results, key=lambda r: r["config_idx"])
indices = [r["config_idx"] for r in sorted_results]
val_l1s = [r["val_l1"] for r in sorted_results]
lrs = [r["learning_rate"] for r in sorted_results]

lr_colors = {1e-4: "#1f77b4", 2e-4: "#ff7f0e", 5e-4: "#2ca02c"}
colors = [lr_colors[lr] for lr in lrs]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(indices, val_l1s, color=colors)
ax.set_xlabel("Configuration #")
ax.set_ylabel("Validation L1 Loss")
ax.set_title("Grid Search Results — Validation L1 by Configuration")

legend_elements = [Patch(facecolor=c, label=f"LR={k:.0e}") for k, c in lr_colors.items()]
ax.legend(handles=legend_elements)

best_idx = min(sorted_results, key=lambda r: r["val_l1"])["config_idx"]
ax.axvline(best_idx, color="red", linestyle="--", alpha=0.7, label="Best")
ax.annotate("Best", xy=(best_idx, min(val_l1s)), fontsize=10, color="red",
            ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "bar_chart_val_l1.png", dpi=100)
plt.show()

In [ ]:
# Heatmap: Val L1 averaged over batch sizes, as (LR × λ_L1)
heatmap_data = {}
for r in grid_results:
    key = (r["learning_rate"], r["lambda_l1"])
    heatmap_data.setdefault(key, []).append(r["val_l1"])

lr_vals = sorted(set(r["learning_rate"] for r in grid_results))
lam_vals = sorted(set(r["lambda_l1"] for r in grid_results))

heat = np.zeros((len(lr_vals), len(lam_vals)))
for i, lr in enumerate(lr_vals):
    for j, lam in enumerate(lam_vals):
        heat[i, j] = np.mean(heatmap_data[(lr, lam)])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heat, cmap="YlOrRd_r", aspect="auto")
ax.set_xticks(range(len(lam_vals)))
ax.set_xticklabels([str(int(v)) for v in lam_vals])
ax.set_yticks(range(len(lr_vals)))
ax.set_yticklabels([f"{v:.0e}" for v in lr_vals])
ax.set_xlabel("λ_L1")
ax.set_ylabel("Learning Rate")
ax.set_title("Average Val L1 (lower = better)")

for i in range(len(lr_vals)):
    for j in range(len(lam_vals)):
        ax.text(j, i, f"{heat[i, j]:.4f}", ha="center", va="center", fontsize=10)

fig.colorbar(im)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "heatmap_lr_lambda.png", dpi=100)
plt.show()

In [ ]:
# Sample grid from top-3 configs
top3 = sorted(grid_results, key=lambda r: r["val_l1"])[:3]

for rank, r in enumerate(top3, 1):
    img_path = OUTPUT_DIR / f"config_{r['config_idx']:02d}.png"
    if img_path.exists():
        img = Image.open(img_path)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.imshow(img)
        ax.set_title(f"Rank #{rank} — Config {r['config_idx']}  "
                     f"(LR={r['learning_rate']:.0e}, BS={r['batch_size']}, λ={r['lambda_l1']}, "
                     f"Val L1={r['val_l1']:.4f})")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

## 8. Takeaways

### Takeaways
- **Best overall configuration:** `Config #24` with **LR = `5e-4`**, **Batch Size = `32`**, **`λ_L1 = 150`**, achieving **Val L1 = `0.209585`**.
- The **top 3 configs** all used higher learning rate (`5e-4`) and stronger reconstruction weighting (`λ_L1 >= 100`), indicating this task benefits from stronger optimization and pixel-level guidance.
- Average validation L1 improved as learning rate increased: `1e-4` (`0.237800`) -> `2e-4` (`0.231832`) -> `5e-4` (`0.226712`).
- Higher `λ_L1` also improved performance on average: `50` (`0.241712`) -> `100` (`0.229202`) -> `150` (`0.225430`).
- Batch size `32` produced the best overall average (`0.230217`) compared with `16` (`0.233302`) and `64` (`0.232826`).